In [1]:
import os
import re
import math
import shutil
import subprocess

from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from matplotlib.dates import DateFormatter
from dateutil.tz import tzutc, tzlocal
from scipy import stats
from scipy.optimize import brentq, curve_fit, fsolve, least_squares



import sys
import signal
import tempfile
import time
from collections import Counter
from IPython.display import clear_output

In [ ]:
%load_ext autoreload
%autoreload 2

import official_GSSHA_Python_functions as gf

%load_ext autoreload
%autoreload 2

import official_GSSHA_Python_post_process_functions as gppf

You will need to download peak water levels from the gauge's historical data. You can find this data on the USGS website of the gauge. Keep in mind, the record should only consist of the rating curve representative of the DEM. This DEM is from 2013, so the curve closest to 2013 is most representative. If the datum was changed before or after 2013, that data should be corrected.  You can use the stream stats tool to quickly view the gauge plot and the notes in the peak water level data to determine when the datum shifted. For waialua, the gauge datum was updated in 2005 (so everything 2006 and onwards is in the current datum).


link for waialua gauge peak water level data : https://waterdata.usgs.gov/monitoring-location/USGS-16210500/#dataTypeId=continuous-00065-0&period=P7D&showFieldMeasurements=true


link to stream stats for waialua gauge: https://streamstats.usgs.gov/ss/?gage=16210500&tab=plots

In [3]:
cwd = os.getcwd()
script_dir = Path.cwd() 

GSSHA_prj_name = "Waialua_FIM_testing"
model_dir = script_dir / GSSHA_prj_name
GSSHA_executables = script_dir / 'GSSHA_applications'
test_input_flow_results_dir = model_dir / "TEST_INPUT_FLOW_SIMULATIONS"

In [4]:
# Once Offset is determined, set up model paramters for running stage height maps
gage_readings = np.arange(26.0, 38.0, 1).tolist()
offset = 13.835 #USGS determined, 12.0 rating curve determined, 15.69 without correction

In [5]:
#Set up the storage of results
modeling_set_test_results= {"gauge_readings_ft":[],
                            "expected_WSE_m":[],
                            "expected_WSE_ft":[],  
                            "convergence_time": [], 
                            "input_flows":[]}

#Fill in some metadata about gauge readings, expected model outputs to figure out the rest
expected_model_outputs = [x - offset for x in gage_readings]
modeling_set_test_results["gauge_readings_ft"] = gage_readings
modeling_set_test_results["expected_WSE_ft"] = expected_model_outputs
expected_WSE_m = [round(ft * 0.3048, 2) for ft in expected_model_outputs]
modeling_set_test_results["expected_WSE_m"] = expected_WSE_m

In [6]:
df_USGS_rating_curve = pd.read_csv("peaks-USGS-16210500.csv")


#only evaluate post 2006 data (gauge was installed in 2012) and only need discharge and stage
df_USGS_rating_curve = (
    df_USGS_rating_curve
    .sort_values("water_year")
    .set_index("water_year", drop=False)
)

df_USGS_rating_curve = df_USGS_rating_curve.loc[2006:]
df_USGS_discharge = df_USGS_rating_curve[
    df_USGS_rating_curve["parameter_code"] == 60
].copy()

df_USGS_stage = df_USGS_rating_curve[
    df_USGS_rating_curve["parameter_code"] == 65
].copy()

df_USGS_rating_curve = (
    df_USGS_discharge[["value"]]
    .rename(columns={"value": "discharge_cfs"})
    .merge(
        df_USGS_stage[["value"]]
        .rename(columns={"value": "stage_ft"}),
        left_index=True,
        right_index=True,
    )
    .reset_index()
)

df_USGS_rating_curve.head()

,water_year,discharge_cfs,stage_ft
0,2006,8530.0,27.64
1,2007,620.0,19.34
2,2008,2530.0,23.38
3,2009,11800.0,29.11
4,2010,1140.0,20.80


In [7]:
#---------------------------------------------------------------
####READ OWS FILE = WATER SURFACE ELEVATION AT GAUGE LOCATION######
#PATHS
ows_file_paths = gppf.get_flow_file_dict(
    test_input_flow_results_dir,
    ".ows"
)

#READING IT
ows_file_read = {}
for flow_cms, filepath in ows_file_paths.items():
    ows = gppf.read_GSSHA_ows(filepath.parent, filepath.name)

    ows_file_read[flow_cms] = ows

#DETERMINING PEAK WATER ELEVATION AT CONSTANT EQUILIBIRUM
peak_wse = {}
for flow_cms, df in ows_file_read.items():
    peak_wse[flow_cms] = df["WSE_m"].tail(120).mean()




#---------------------------------------------------------------
####READ OQC FILE = CUMULATIVE DISCHARGE AT GAUGE LOCATION######
#PATHS
oqc_file_paths = gppf.get_flow_file_dict(
    test_input_flow_results_dir,
    ".oqc"
)

#READING IT
oqc_file_read = {}
for flow_cms, filepath in oqc_file_paths.items():
    oqc = gppf.read_GSSHA_oqc(filepath)

    oqc_file_read[flow_cms] = oqc


#DETERMINING DISCHARGE AT STABLE EQUILIBIRUM CONDITIONS
oqc_stable_discharge = {} #max discharge recorded at gauge
for flow_cms, df in oqc_file_read.items():
    # Calculate instantaneous discharge
    df = gppf.cumulative_to_instantaneous(df)

    # Save the updated DataFrame back to the dictionary
    oqc_file_read[flow_cms] = df

    # Average of the last 60 minutes
    oqc_stable_discharge[flow_cms] = (
        df["instantaneous_discharge"].tail(120).median()
    )

####MERGE DATA INTO CLEAN ORGANIZED FORMAT FOR FURTHER PROCESSING
rating_curve_data = {
    flow_cms: {
        "discharge_cms": oqc_stable_discharge[flow_cms],
        "stage_m": peak_wse[flow_cms],
    }
    for flow_cms in peak_wse
}

NameError: name 'gppf' is not defined

In [ ]:
#CREATE STAGE TO DISCHARGE RATING CURVE of model!
stage_model = gppf.fit_rating_curve(rating_curve_data)

In [ ]:
# Extract data
discharge = np.array(
    [
        rating_curve_data[flow]["discharge_cms"]
        for flow in sorted(rating_curve_data)
    ],
    dtype=float,
)

stage = np.array(
    [
        rating_curve_data[flow]["stage_m"]
        for flow in sorted(rating_curve_data)
    ],
    dtype=float,
)

# Smooth discharge values
discharge_smooth = np.linspace(
    0,
    discharge.max() * 1.25,
    1000,
)

# Predicted stage from fitted rating curve
stage_smooth = gppf.rating_curve_equation(
    discharge_smooth,
    stage_model["parameters"]["a"],
    stage_model["parameters"]["b"],
)

# ±10% uncertainty bounds
uncertainty_fraction = 0.10

stage_lower = stage_smooth * (1 - uncertainty_fraction)
stage_upper = stage_smooth * (1 + uncertainty_fraction)


# Plot
plt.figure(figsize=(8, 6))

plt.scatter(
    discharge,
    stage,
    s=60,
    label="Model results",
    zorder=3,
)

plt.plot(
    discharge_smooth,
    stage_smooth,
    linewidth=2.5,
    label="Rating curve",
)

plt.fill_between(
    discharge_smooth,
    stage_lower,
    stage_upper,
    alpha=0.25,
    label="±10% uncertainty",
)

plt.xlabel("Gauge Discharge (cms)")
plt.ylabel("Stage (m)")
plt.title("Stage–Discharge Rating Curve")

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(stage_model["equation"])
print(f'R² = {stage_model["r_squared"]:.5f}')

In [ ]:
df_USGS_rating_curve["discharge_cms"] = (
    df_USGS_rating_curve["discharge_cfs"] * 0.028316846592
)

df_USGS_rating_curve["stage_m"] = (
    df_USGS_rating_curve["stage_ft"] * 0.3048
)

usgs_predicted_stage_m = gppf.rating_curve_equation(
    df_USGS_rating_curve["discharge_cms"].to_numpy(),
    stage_model["parameters"]["a"],
    stage_model["parameters"]["b"],
)

stage_offset_m = np.mean(
    df_USGS_rating_curve["stage_m"].to_numpy()
    - usgs_predicted_stage_m
)

print(f"Optimal vertical offset: {stage_offset_m:.4f} m")
print(f"Optimal vertical offset: {stage_offset_m / 0.3048:.4f} ft")

In [ ]:
# Fitted rating-curve parameters
a = stage_model["parameters"]["a"]
b = stage_model["parameters"]["b"]

# USGS observations
Q_usgs = df_USGS_rating_curve["discharge_cms"].to_numpy()
stage_usgs = df_USGS_rating_curve["stage_m"].to_numpy()

# Smooth discharge values for plotting the modeled curves
Q_plot = np.linspace(
    0,
    max(
        Q_usgs.max(),
        stage_model["discharge"].max(),
    ) * 1.1,
    1000,
)

# Original modeled rating curve
stage_original = gppf.rating_curve_equation(
    Q_plot,
    a,
    b,
)

# Predicted modeled stage at USGS discharge values
stage_model_at_usgs = gppf.rating_curve_equation(
    Q_usgs,
    a,
    b,
)

# Mean vertical offset
stage_offset_m = np.mean(
    stage_usgs - stage_model_at_usgs
)

# ±10% uncertainty applied to modeled stage
uncertainty_fraction = 0.10

stage_offset_lower_m = np.mean(
    stage_usgs
    - stage_model_at_usgs * (1 + uncertainty_fraction)
)

stage_offset_upper_m = np.mean(
    stage_usgs
    - stage_model_at_usgs * (1 - uncertainty_fraction)
)

# Shifted modeled curves
stage_shifted = stage_original + stage_offset_m

stage_shifted_lower = (
    stage_original + stage_offset_lower_m
)

stage_shifted_upper = (
    stage_original + stage_offset_upper_m
)


plt.figure(figsize=(9, 6))

# Original model curve
plt.plot(
    Q_plot,
    stage_original,
    linewidth=2,
    label="Original modeled rating curve",
)

# Vertically shifted model curve
plt.plot(
    Q_plot,
    stage_shifted,
    linewidth=2,
    linestyle="--",
    label=f"Vertically shifted model ({stage_offset_m:+.3f} m)",
)

# Uncertainty band around shifted model
plt.fill_between(
    Q_plot,
    stage_shifted_lower,
    stage_shifted_upper,
    alpha=0.25,
    label="±10% offset uncertainty",
)

# Original USGS stage-discharge points
plt.scatter(
    Q_usgs,
    stage_usgs,
    s=40,
    label="USGS rating-curve points",
    zorder=3,
)

plt.xlabel("Discharge (m³/s)")
plt.ylabel("Stage (m)")
plt.title("Modeled and USGS Stage–Discharge Rating Curves")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


print(
    f"Mean offset: "
    f"{stage_offset_m:.4f} m "
    f"({stage_offset_m / 0.3048:.4f} ft)"
)

print(
    f"Lower offset bound: "
    f"{stage_offset_lower_m:.4f} m "
    f"({stage_offset_lower_m / 0.3048:.4f} ft)"
)

print(
    f"Upper offset bound: "
    f"{stage_offset_upper_m:.4f} m "
    f"({stage_offset_upper_m / 0.3048:.4f} ft)"
)